# NLP – Task 2
## Text Classification and Model Comparison

### Objective
Implement and fairly compare five machine learning and deep learning models:
1. Prepare the Shared Dataset 
2. Prepare Text Features 
3. Support Vector Machine (SVM)
4. Naive Bayes
5. Simple RNN
6. LSTM
7. GRU
8. Evaluate Every Model  
9. Compare the Five Models 
10. Predict Unseen Text 

The project continues the preprocessing workflow from NLP Task 1.

All models will use the same cleaned dataset and the same stratified
80/20 train-test split with random_state=3.

## Project Setup

In [73]:
import sys

print(sys.executable)
print(sys.version)

c:\samawork\uni\nlp\task2\.venv\Scripts\python.exe
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


### Install Required Libraries

In [74]:
%pip install pandas numpy matplotlib seaborn scikit-learn nltk tensorflow

Note: you may need to restart the kernel to use updated packages.


### Import Libraries

In [75]:
# Basic libraries
import os
import re
import time
import random
import warnings

# Data manipulation
import numpy as np
import pandas as pd

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Natural Language Processing
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Deep Learning
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Embedding,
    SimpleRNN,
    LSTM,
    GRU,
    Dense,
    Dropout
)

warnings.filterwarnings("ignore")

print("All libraries imported successfully!")

All libraries imported successfully!


### Set Random Seeds

In [76]:
SEED = 3

random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf

tf.keras.utils.set_random_seed(SEED)

print("Random seed:", SEED)

Random seed: 3


### Download NLTK Resources

In [77]:
# Download NLTK Resources

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
print("NLTK resources are ready!")

NLTK resources are ready!


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\samae\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\samae\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\samae\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\samae\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\samae\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\samae\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## 1. Prepare the Shared Dataset

### Load Dataset

In [78]:
#Load IMDb Dataset

from pathlib import Path

DATA_DIR = Path("data")

reviews = []
sentiments = []

for split in ["train", "test"]:
    for label in ["pos", "neg"]:

        folder = DATA_DIR / split / label

        for file_path in folder.glob("*.txt"):

            with open(file_path, "r", encoding="utf-8", errors="replace") as file:
                review = file.read()

            reviews.append(review)
            sentiments.append(label)

# Create the shared DataFrame
df = pd.DataFrame({
    "statement": reviews,
    "status": sentiments
})

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset shape: (50000, 2)


,statement,status
0,Bromwell High is a cartoon comedy. It ran at t...,pos
1,Homelessness (or Houselessness as George Carli...,pos
2,Brilliant over-acting by Lesley Ann Warren. Be...,pos
3,This is easily the most underrated film inn th...,pos
4,This is not the typical Mel Brooks film. It wa...,pos


### Inspect Dataset

In [79]:
#Inspect Dataset

print("Dataset Information:")
print("-" * 50)

print("Shape:", df.shape)

print("\nColumn Data Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nNumber of Duplicate Rows:")
print(df.duplicated().sum())

print("\nClass Distribution:")
print(df["status"].value_counts())

print("\nClass Distribution Percentage:")
print(df["status"].value_counts(normalize=True) * 100)

Dataset Information:
--------------------------------------------------
Shape: (50000, 2)

Column Data Types:
statement    str
status       str
dtype: object

Missing Values:
statement    0
status       0
dtype: int64

Number of Duplicate Rows:
418

Class Distribution:
status
pos    25000
neg    25000
Name: count, dtype: int64

Class Distribution Percentage:
status
pos    50.0
neg    50.0
Name: proportion, dtype: float64


### Select Required Columns

In [80]:
required_columns = ["statement", "status"]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

df = df[required_columns].copy()

print("Selected columns:", df.columns.tolist())
display(df.head())

Selected columns: ['statement', 'status']


,statement,status
0,Bromwell High is a cartoon comedy. It ran at t...,pos
1,Homelessness (or Houselessness as George Carli...,pos
2,Brilliant over-acting by Lesley Ann Warren. Be...,pos
3,This is easily the most underrated film inn th...,pos
4,This is not the typical Mel Brooks film. It wa...,pos


### Handle Missing Values and Duplicates

In [81]:
print("Original shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

df = df.dropna(
    subset=["statement", "status"]
).copy()

df["statement"] = df["statement"].astype(str).str.strip()

df["status"] = df["status"].astype(str).str.strip()

df = df[
    (df["statement"] != "") &
    (df["status"] != "")
].copy()

df = df.drop_duplicates(
    subset=["statement", "status"]
).reset_index(drop=True)

print("\nShape after cleaning:", df.shape)

print("\nRemaining missing values:")
print(df.isnull().sum())

print("\nRemaining duplicates:", df.duplicated().sum())

Original shape: (50000, 2)

Missing values:
statement    0
status       0
dtype: int64

Shape after cleaning: (49582, 2)

Remaining missing values:
statement    0
status       0
dtype: int64

Remaining duplicates: 0


### Merge Suicidal into Depression

In [82]:
print("Classes before merging:")
print(df["status"].value_counts())

df["status"] = df["status"].str.strip().str.lower()

df["status"] = df["status"].replace({
    "suicidal": "depression"
})

print("\nClasses after merging:")
print(df["status"].value_counts())

print("\nUnique classes:")
print(df["status"].unique())

Classes before merging:
status
pos    24884
neg    24698
Name: count, dtype: int64

Classes after merging:
status
pos    24884
neg    24698
Name: count, dtype: int64

Unique classes:
<StringArray>
['pos', 'neg']
Length: 2, dtype: str


### 2. Text Preprocessing

In [83]:
#  Text Preprocessing Function

stop_words = set(stopwords.words("english"))

negation_words = {"not", "no", "never"}
stop_words = stop_words - negation_words

lemmatizer = WordNetLemmatizer()


def preprocess_text(text):

    text = str(text).lower()

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove punctuation while preserving apostrophes
    text = re.sub(r"[^\w\s']", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords while preserving negation
    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    # Lemmatization
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    return " ".join(tokens)


df["processed_text"] = df["statement"].apply(
    preprocess_text
)

# Remove empty processed texts before splitting
df = df[
    df["processed_text"].str.strip() != ""
].copy()

df = df.reset_index(drop=True)

print("Preprocessing completed!")

display(
    df[["statement", "processed_text", "status"]].head()
)

Preprocessing completed!


,statement,processed_text,status
0,Bromwell High is a cartoon comedy. It ran at t...,bromwell high cartoon comedy ran time program ...,pos
1,Homelessness (or Houselessness as George Carli...,homelessness houselessness george carlin state...,pos
2,Brilliant over-acting by Lesley Ann Warren. Be...,brilliant acting lesley ann warren best dramat...,pos
3,This is easily the most underrated film inn th...,easily underrated film inn brook cannon sure f...,pos
4,This is not the typical Mel Brooks film. It wa...,not typical mel brook film much less slapstick...,pos


### Verify Negation Preservation

In [84]:
test_sentences = [
    "I am not happy",
    "I have no hope",
    "I never feel good"
]

for sentence in test_sentences:
    print("Original:", sentence)
    print("Processed:", preprocess_text(sentence))
    print("-" * 50)

Original: I am not happy
Processed: not happy
--------------------------------------------------
Original: I have no hope
Processed: no hope
--------------------------------------------------
Original: I never feel good
Processed: never feel good
--------------------------------------------------


### Train-Test Split

Split the dataset into 80% training and 20% testing.

Use stratification to preserve the class distribution.

random_state = 3

The same train and test samples will be used by all five models.

In [85]:
X = df["processed_text"]
y = df["status"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=3,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining distribution:")
print(y_train.value_counts())

print("\nTesting distribution:")
print(y_test.value_counts())

print("\nTrain indices:", X_train.index[:10].tolist())
print("Test indices:", X_test.index[:10].tolist())

Training samples: 39665
Testing samples: 9917

Training distribution:
status
pos    19907
neg    19758
Name: count, dtype: int64

Testing distribution:
status
pos    4977
neg    4940
Name: count, dtype: int64

Train indices: [21140, 6429, 28211, 18705, 41368, 42910, 32990, 24524, 43954, 49536]
Test indices: [3538, 26353, 5392, 39195, 39937, 11262, 7003, 23906, 9850, 39609]


### Encode Target Labels

In [86]:
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)

y_test_encoded = label_encoder.transform(y_test)

class_names = label_encoder.classes_

num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)

print("\nEncoded training labels:")
print(y_train_encoded[:10])

print("\nEncoded testing labels:")
print(y_test_encoded[:10])

Classes: ['neg' 'pos']
Number of classes: 2

Encoded training labels:
[0 1 1 0 0 0 1 0 0 0]

Encoded testing labels:
[1 1 1 0 0 1 1 0 1 0]


## 2. Prepare Text Features

### TF-IDF

In [87]:
MAX_FEATURES = 10000

tfidf_vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TF-IDF Training Shape:", X_train_tfidf.shape)
print("TF-IDF Testing Shape:", X_test_tfidf.shape)

print("Vocabulary Size:", len(tfidf_vectorizer.vocabulary_))

TF-IDF Training Shape: (39665, 10000)
TF-IDF Testing Shape: (9917, 10000)
Vocabulary Size: 10000


### Tokenizer & Padding

In [88]:
MAX_WORDS = 10000
MAX_LENGTH = 200
EMBEDDING_DIM = 64

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

train_sequences = tokenizer.texts_to_sequences(X_train)

test_sequences = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    train_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    test_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

print("Vocabulary Size:", len(tokenizer.word_index))

print("Training Shape:", X_train_pad.shape)

print("Testing Shape:", X_test_pad.shape)

Vocabulary Size: 84098
Training Shape: (39665, 200)
Testing Shape: (9917, 200)


## 3. Support Vector Machine (SVM)

### Initialize Support Vector Machine

In [ ]:
svm_model = LinearSVC()

print("SVM model initialized successfully!")

SVM model initialized successfully!


### Train SVM Model

In [ ]:
start_time = time.time()

svm_model.fit(
    X_train_tfidf,
    y_train
)

svm_train_time = time.time() - start_time

print("SVM training completed successfully!")
print(f"Training Time: {svm_train_time:.4f} seconds")

SVM training completed successfully!
Training Time: 0.9534 seconds


### Generate SVM Predictions

In [ ]:
svm_predictions = svm_model.predict(X_test_tfidf)

print("SVM predictions generated successfully!")

print("\nFirst 10 Predictions:")
print(svm_predictions[:10])

print("\nFirst 10 Actual Labels:")
print(y_test.values[:10])

SVM predictions generated successfully!

First 10 Predictions:
['pos' 'pos' 'pos' 'neg' 'neg' 'pos' 'pos' 'neg' 'pos' 'pos']

First 10 Actual Labels:
<StringArray>
['pos', 'pos', 'pos', 'neg', 'neg', 'pos', 'pos', 'neg', 'pos', 'neg']
Length: 10, dtype: str


### Compare Actual and Predicted Labels

In [ ]:
svm_results = pd.DataFrame({
    "Review": X_test.values[:10],
    "Actual Sentiment": y_test.values[:10],
    "Predicted Sentiment": svm_predictions[:10]
})

display(svm_results)

,Review,Actual Sentiment,Predicted Sentiment
0,sunday july 16 8 00pm castro san francisco sou...,pos,pos
1,year puzzled julie brown west coast n't househ...,pos,pos
2,went wal mart found film dvd no idea whether m...,pos,pos
3,watched acacia find agree negative review like...,neg,neg
4,shocked 18 page good review one worst movie es...,neg,neg
5,sure 's 50 's drive special n't let fool littl...,pos,pos
6,movie wonderful show man 's work wilderness na...,pos,pos
7,dont ever ever ever consider watching sorry ex...,neg,neg
8,kurt russell elvis plain simple dedication rol...,pos,pos
9,woody allen seems completely devoid inspiratio...,neg,pos
